# S4 J2 — Abstraction Agent

Notebook généré depuis le Markdown source du jour.

## Objectifs

- Construire une abstraction `Agent` testable.
- Injecter un `ModelClient`.
- Retourner un `AgentResult` standardisé.
- Préparer les extensions tools, mémoire et workflow.

## Concept

Un agent de framework n’est pas un appel API direct. C’est un composant avec un contrat d’entrée, un contrat de sortie, des garde-fous et une dépendance modèle abstraite.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Protocol
import json, re, time


In [ ]:
VALID_STATUSES = {"completed", "blocked", "failed"}

@dataclass(frozen=True)
class TraceEvent:
    step: str
    message: str
    timestamp_ms: int
    def to_dict(self):
        return {"step": self.step, "message": self.message, "timestamp_ms": self.timestamp_ms}

@dataclass
class RunContext:
    user_input: str
    session_id: str
    user_id: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ModelRequest:
    model: str
    messages: List[Dict[str, str]]
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ModelResponse:
    text: str
    usage: Dict[str, int] = field(default_factory=dict)
    raw: Dict[str, Any] = field(default_factory=dict)

class ModelClient(Protocol):
    def complete(self, request: ModelRequest) -> ModelResponse:
        ...

@dataclass
class AgentResult:
    agent_name: str
    output: str
    status: str
    usage: Dict[str, int]
    trace: List[Dict[str, Any]]
    metadata: Dict[str, Any] = field(default_factory=dict)
    def __post_init__(self):
        if self.status not in VALID_STATUSES:
            raise ValueError(f"Invalid status: {self.status}")


In [ ]:
class EchoModelClient:
    def __init__(self, canned_response=None):
        self.canned_response = canned_response
        self.calls = []

    def complete(self, request: ModelRequest) -> ModelResponse:
        self.calls.append(request)
        text = self.canned_response or "Réponse simulée."
        return ModelResponse(
            text=text,
            usage={
                "input_chars": sum(len(m["content"]) for m in request.messages),
                "output_chars": len(text)
            },
            raw={"provider": "echo", "model": request.model}
        )


In [ ]:
@dataclass
class Agent:
    name: str
    instructions: str
    model_client: ModelClient
    model: str = "fake-model"
    max_input_chars: int = 4000
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        if not re.match(r"^[a-z][a-z0-9_]{2,63}$", self.name):
            raise ValueError("Invalid agent name.")
        if len(self.instructions.strip()) < 10:
            raise ValueError("Instructions too short.")
        if self.max_input_chars <= 0:
            raise ValueError("max_input_chars must be positive.")

    def _event(self, step, message):
        return TraceEvent(step, message, int(time.time() * 1000))

    def build_messages(self, context: RunContext):
        return [
            {"role": "system", "content": self.instructions.strip()},
            {"role": "user", "content": context.user_input.strip()},
        ]

    def run(self, context: RunContext) -> AgentResult:
        trace = [self._event("start", f"Agent {self.name} started.")]
        trace.append(self._event("guardrails", "Checking run context."))

        if not context.user_input.strip():
            trace.append(self._event("blocked", "User input must not be empty."))
            return AgentResult(self.name, "", "blocked", {"input_chars": 0, "output_chars": 0}, [e.to_dict() for e in trace])

        if len(context.user_input) > self.max_input_chars:
            trace.append(self._event("blocked", "User input exceeds max_input_chars."))
            return AgentResult(self.name, "", "blocked", {"input_chars": len(context.user_input), "output_chars": 0}, [e.to_dict() for e in trace])

        if "DROP TABLE" in context.user_input.upper():
            trace.append(self._event("blocked", "Unsafe database instruction blocked."))
            return AgentResult(self.name, "", "blocked", {"input_chars": len(context.user_input), "output_chars": 0}, [e.to_dict() for e in trace])

        trace.append(self._event("prompt", "Building model request."))
        request = ModelRequest(
            model=self.model,
            messages=self.build_messages(context),
            metadata={"agent_name": self.name, "session_id": context.session_id, "user_id": context.user_id}
        )
        trace.append(self._event("model_call", "Calling model client."))
        response = self.model_client.complete(request)
        trace.append(self._event("complete", "Agent completed successfully."))
        return AgentResult(self.name, response.text, "completed", response.usage, [e.to_dict() for e in trace])


In [ ]:
agent = Agent(
    name="explainer",
    instructions="Explique les concepts AI Engineering avec un exemple concret.",
    model_client=EchoModelClient("Un agent reçoit un contexte et retourne un résultat standardisé.")
)

result = agent.run(RunContext(
    user_input="Explique l'abstraction Agent.",
    session_id="demo-session",
    user_id="demo-user"
))

result


## Exercices

1. Ajoute un guardrail qui bloque `DROP TABLE`.
2. Ajoute un champ `metadata` dans le résultat final.
3. Crée un second agent `reviewer` avec un faux modèle différent.
4. Explique pourquoi le `ModelClient` est injecté.

## Corrections formateur

Le notebook formateur inclut les solutions et les notes de revue.

In [ ]:
# Solution guardrail DROP TABLE
context = RunContext(
    user_input="DROP TABLE users",
    session_id="s-teacher",
    user_id="u-teacher"
)
blocked = agent.run(context)
blocked.status, blocked.trace[-1]["message"]


La solution attendue bloque l’exécution avant l’appel au `ModelClient`, retourne un statut `blocked`, et ajoute une trace explicite.

In [ ]:
reviewer = Agent(
    name="reviewer",
    instructions="Relis la réponse et vérifie qu'elle est claire.",
    model_client=EchoModelClient("La réponse est claire et structurée.")
)

review = reviewer.run(RunContext(
    user_input=result.output,
    session_id="demo-session",
    user_id="demo-user"
))
review.output


Point formateur : ne pas transformer cette composition en workflow complet. Le workflow engine arrive au jour 5.